# R7.9 | 卫星 ADT 与 GLORYS 流轴的 D(t) 三联图

**完整单 Notebook；WSL / VS Code；不训练、不重新追踪 ADT。**

- 图(a)：逐日两条流轴的双向最近路径**平均**距离 `D(t)`（直方图/KDE）；
- 图(b)：六条断面交点**径向距离差**的分布（箱线图）；
- 图(c)：逐日 `D(t)`、90天展示平滑及年度变化；另输出六断面时间变化六联图。

默认沿轴线弧长做轻度空间平滑，避免锯齿状路径点；**不改原 NPZ**；保留每日未平滑共域平均距离用于敏感性检查。仅使用 R7.2 审计通过的全轨迹日期，2023 不完整轨迹不混入。

**注意：** 这不是模型的 RMSE。此前 R7.2 `42.04 km` 是共域双向最近路径 **RMSE**；这里 `D(t)` 是逐日平均最近距离（MAE 类量），不要把两者当作同一指标。

In [ ]:
# 第 1 步：Python 依赖核查。推荐在VS Code的WSL解释器中执行。
import sys, importlib.util
for pkg in ['numpy','pandas','scipy','matplotlib']:
    print(pkg, 'OK' if importlib.util.find_spec(pkg) else 'MISSING')
print('Python:',sys.executable)


## 第 2 步：完整分析模块（已嵌入，无外部 .py 依赖）

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""R7.9: ADT-vs-GLORYS Kuroshio axis spatial-distance three-panel figures.

Inputs: frozen daily NPZ from R70 and accepted dates from R72 audit.
No training, no ADT retracking, no alteration of source NPZ.
"""
from __future__ import annotations
import io
import os
import json
import sys
import csv
import math
import hashlib
import zipfile
import warnings
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree
from scipy.ndimage import gaussian_filter1d
from scipy.stats import gaussian_kde
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib import font_manager

EARTH_KM = 6371.0088
SECTIONS = {
    'A': ((121.50, 23.50), (125.00, 22.75)),
    'B': ((121.50, 26.25), (124.25, 24.25)),
    'PN': ((126.00, 29.00), (128.15, 27.30)),
    'C': ((130.25, 31.25), (132.50, 27.00)),
    'D': ((137.00, 34.75), (138.00, 27.75)),
    'E': ((141.75, 40.00), (145.00, 31.50)),
}
SECTION_ORDER = list(SECTIONS)

@dataclass
class Config:
    ROOT: str = os.environ.get('KUROSHIO_ROOT', '/mnt/d/KuroshioDaily1993_2025')
    MODE: str = 'smooth_npz'  # 'smooth_npz': recompute on frozen NPZ; 'archive_fast': old unsmoothed CSV
    OUT_REL: str = 'R79_ADT_GLORYS_SmoothedDistance_ThreePanel'
    ADT_REL: str = 'R70_ADT_Independent_Python/adt_axes_npz'
    GLORYS_REL: str = 'final/tracks_final'
    R70_RESULT_NAME: str = 'R70_PY_上传给ChatGPT_ADT独立参考验证完整结果.zip'
    R72_RESULT_NAME: str = 'R72_上传给ChatGPT_全量共域与2023断面审计.zip'
    DOMAIN_EAST: float = 144.3
    RESAMPLE_SPACING_KM: float = 5.0
    SPATIAL_SMOOTH_SIGMA_KM: float = 10.0
    MAX_SMOOTH_DISPLACEMENT_KM: float = 12.0
    MIN_AXIS_POINTS: int = 10
    STRICT_FILES: bool = True
    # All qualifying dates in R72; for a smoke run set e.g. 24, then None for official output
    MAX_DAYS: int | None = None
    MIN_MONTH_DAYS: int = 15
    MIN_ANNUAL_DAYS: int = 180
    DISPLAY_ROLL_DAYS: int = 90
    DISPLAY_ROLL_MIN_DAYS: int = 45
    LANG: str = 'zh'  # 'zh' or 'en'
    DPI: int = 240
    ZIP_RESULTS: bool = True


def unit_xyz(axis):
    a=np.deg2rad(np.asarray(axis, dtype=float))
    lo,la=a[:,0],a[:,1]
    return np.column_stack([np.cos(la)*np.cos(lo), np.cos(la)*np.sin(lo), np.sin(la)])


def spherical_km_from_chord(chord):
    return 2.0*EARTH_KM*np.arcsin(np.clip(np.asarray(chord)/(2.0), 0,1))


def segment_km(axis):
    u=unit_xyz(axis)
    return spherical_km_from_chord(np.linalg.norm(np.diff(u,axis=0),axis=1))


def spherical_dist_km(p,q):
    return float(spherical_km_from_chord(np.linalg.norm(unit_xyz(np.asarray([p]))[0]-unit_xyz(np.asarray([q]))[0])))


def validate_axis(axis, desc='axis'):
    x=np.asarray(axis,dtype=float)
    if x.ndim != 2 or x.shape[1] != 2 or len(x) < 3:
        raise ValueError(f'{desc}: invalid shape {x.shape}')
    if not np.isfinite(x).all():raise ValueError(f'{desc}: contains NaN/Inf')
    if not ((x[:,0] >= 110)&(x[:,0] <= 155)&(x[:,1] >= 15)&(x[:,1] <= 45)).all():
        raise ValueError(f'{desc}: coordinates outside regional safety bounds')
    step=segment_km(x)
    if (step>250).any():raise ValueError(f'{desc}: discontinuity >250 km')
    # Duplicate coincident knots are safe to drop, not interpolate across invalid stretches.
    keep=np.r_[True, step>=.02]
    x=x[keep]
    if len(x)<3:raise ValueError(f'{desc}: too few distinct points')
    return x


def load_axis(path, product):
    with np.load(path,allow_pickle=False) as z:
        lk='axis_lon' if product=='ADT' else 'lon'
        ak='axis_lat' if product=='ADT' else 'lat'
        if lk not in z or ak not in z:
            raise KeyError(f'{path}: missing keys {lk}/{ak}; present {z.files}')
        x=np.column_stack([z[lk].reshape(-1),z[ak].reshape(-1)])
    return validate_axis(x,f'{product} {path.name}')


def clip_first_east_cross(axis, east=144.3):
    x=np.asarray(axis,float)
    for i in range(1,len(x)):
        a,b=x[i-1,0],x[i,0]
        if a < east <= b:
            f=(east-a)/(b-a)
            last=x[i-1]+f*(x[i]-x[i-1]);last[0]=east
            return validate_axis(np.vstack([x[:i],last]),'clipped axis')
    raise ValueError(f'axis does not cross {east:.1f}E eastward')


def resample_arclength(axis, spacing_km=5.0):
    x=validate_axis(axis,'pre-resampling axis')
    s=np.r_[0.,np.cumsum(segment_km(x))]
    if s[-1] < 25.0:raise ValueError('axis shorter than 25 km')
    n=max(2,int(math.ceil(s[-1]/spacing_km))+1)
    new_s=np.linspace(0,s[-1],n)
    return np.column_stack([np.interp(new_s,s,x[:,0]),np.interp(new_s,s,x[:,1])])


def light_spatial_smooth(axis, sigma_km=10.0, spacing_km=5.0, max_move_km=12.0):
    """Symmetric, arclength-Gaussian smoothing (not time smoothing).

    Geographic xyz smoothing avoids longitude/latitude axis distortions. Norm-to-sphere
    and cap point movement, keeping both path endpoints EXACTLY unchanged.
    """
    x=resample_arclength(axis,spacing_km)
    if sigma_km <= 0:return x,0.0
    u=unit_xyz(x)
    smooth=np.column_stack([gaussian_filter1d(u[:,j],sigma_km/spacing_km,mode='nearest') for j in range(3)])
    smooth /= np.maximum(np.linalg.norm(smooth,axis=1,keepdims=True),1e-15)
    delta=spherical_km_from_chord(np.linalg.norm(u-smooth,axis=1))
    alpha=np.minimum(1.0,max_move_km/np.maximum(delta,1e-9))
    mixed=u*(1-alpha[:,None])+smooth*alpha[:,None]
    mixed /= np.linalg.norm(mixed,axis=1,keepdims=True)
    lon=np.rad2deg(np.arctan2(mixed[:,1],mixed[:,0]))
    lat=np.rad2deg(np.arctan2(mixed[:,2],np.hypot(mixed[:,0],mixed[:,1])))
    result=np.column_stack([lon,lat]);result[[0,-1]]=x[[0,-1]]
    actual=float(np.max(spherical_km_from_chord(np.linalg.norm(u-unit_xyz(result),axis=1))))
    return validate_axis(result,'smoothed axis'),actual


def closest_distance_metrics(a,b):
    pa=unit_xyz(a);pb=unit_xyz(b)
    da=spherical_km_from_chord(cKDTree(pb).query(pa,k=1)[0])
    db=spherical_km_from_chord(cKDTree(pa).query(pb,k=1)[0])
    d=np.r_[da,db]
    return {'D_mean_km':float(d.mean()),'D_RMSE_km':float(np.sqrt(np.mean(d*d))),
            'D_median_km':float(np.median(d)),'D_P90_km':float(np.quantile(d,.9)),
            'samples':len(d)}


def cross2(u,v):
    return u[...,0]*v[...,1]-u[...,1]*v[...,0]


def section_crossings(axis,sec):
    """Return intersection distances along fixed section; explicit ambiguity bookkeeping."""
    p0,p1=SECTIONS[sec]
    lat0=(p0[1]+p1[1])/2
    fx=111.195*np.cos(np.deg2rad(lat0));fy=111.195
    xy=np.asarray(axis,float)*np.array([fx,fy])
    q0=np.asarray(p0)*np.array([fx,fy]);q1=np.asarray(p1)*np.array([fx,fy])
    r=q1-q0;p=xy[:-1];v=np.diff(xy,axis=0)
    den=cross2(v,r);vld=np.abs(den)>1e-10
    diff=q0-p
    t=np.full(len(den),np.nan);q=np.full(len(den),np.nan)
    t[vld]=cross2(diff[vld],r)/den[vld]
    q[vld]=cross2(diff[vld],v[vld])/den[vld]
    ok=vld & (t>=-1e-8)&(t<=1+1e-8)&(q>=-1e-8)&(q<=1+1e-8)
    hits=[]
    for ix in np.flatnonzero(ok):
        loc=axis[ix]+np.clip(t[ix],0,1)*(axis[ix+1]-axis[ix])
        dk=spherical_dist_km(p0,loc)
        if not any(abs(dk-old)<0.5 for old in hits):hits.append(dk)
    return hits


def safe_zip_csv(zpath,inner):
    with zipfile.ZipFile(zpath) as z:
        return pd.read_csv(io.BytesIO(z.read(inner)))


def discover_dates(cfg,root):
    z=root/cfg.R72_RESULT_NAME
    if not z.is_file():
        raise FileNotFoundError('Missing R72 full common-domain archive: '+str(z)+
            '. Copy the original R72 ZIP to the project ROOT (or choose MODE=archive_fast only when both archives exist).')
    d=safe_zip_csv(z,'tables/02_完整共域逐日误差.csv')
    d=d[d.status=='ok'].copy();d.date=pd.to_datetime(d.date,errors='raise')
    if d.date.duplicated().any():raise ValueError('Duplicate dates in R72 valid date registry')
    return d.sort_values('date').reset_index(drop=True)


def prepare_sources(cfg):
    root=Path(cfg.ROOT).expanduser()
    if not root.is_dir():
        raise FileNotFoundError(f'ROOT not found: {root}\n'
            'WSL should mount the Windows D drive at /mnt/d. Colab cannot read your local D: drive. '
            'For ZIP-only plotting put archives in a readable folder and set MODE=archive_fast.')
    out=root/cfg.OUT_REL
    for folder in ('tables','figures','reports'):(out/folder).mkdir(parents=True,exist_ok=True)
    return root,out


def measure_smoothed(cfg,root,out):
    dates=discover_dates(cfg,root)
    if cfg.MAX_DAYS:dates=dates.iloc[:cfg.MAX_DAYS]
    adt=root/cfg.ADT_REL;glo=root/cfg.GLORYS_REL
    if not adt.is_dir() or not glo.is_dir():
        raise FileNotFoundError(f'Missing daily axis NPZ folders: ADT={adt} exists={adt.exists()}; GLORYS={glo} exists={glo.exists()}')
    missing=[]
    for day in dates.date:
        k=day.strftime('%Y%m%d')
        if not (adt/f'{k}.npz').is_file():missing.append((k,'ADT'))
        if not (glo/f'{k}.npz').is_file():missing.append((k,'GLORYS'))
    if missing:
        msg=f'Missing {len(missing)} daily NPZ (first 10: {missing[:10]}); cannot compute smoothed D(t) from R72 CSV.'
        (out/'reports'/'MISSING_FILES.txt').write_text(msg,encoding='utf-8')
        if cfg.STRICT_FILES:raise FileNotFoundError(msg)
        print('[WARNING]',msg)
    oldfile=out/'tables'/'01_smoothed_daily_path_distance.csv'
    oldsec=out/'tables'/'02_smoothed_daily_section_intersections.csv'
    finished=set()
    old_d,old_s=[],[]
    contract=asdict(cfg).copy()
    for k in ('ZIP_RESULTS','DPI','LANG','MAX_DAYS'):contract.pop(k,None)
    chash=hashlib.sha256(json.dumps(contract,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
    cache_sig=out/'reports'/'processing_contract.sha256'
    if oldfile.is_file() and oldsec.is_file() and cache_sig.exists() and cache_sig.read_text().strip()==chash:
        previous_d=pd.read_csv(oldfile);previous_s=pd.read_csv(oldsec)
        finished=set(previous_d.loc[previous_d.status=='ok','date'].astype(str))
        old_d=previous_d.to_dict('records');old_s=previous_s.to_dict('records')
        print(f'[CACHE] {len(finished)} completed dates already loaded')
    elif oldfile.exists() and oldsec.exists():
        print('[CACHE] configuration changed; saved CSV results will be recomputed from immutable NPZ')
    daily=[];section=[]
    for idx,day in enumerate(dates.date,1):
        dayid=day.strftime('%Y-%m-%d');stem=day.strftime('%Y%m%d')
        if dayid in finished:continue
        try:
            xa=load_axis(adt/f'{stem}.npz','ADT');xg=load_axis(glo/f'{stem}.npz','GLORYS')
            xa=clip_first_east_cross(xa,cfg.DOMAIN_EAST)
            xg=clip_first_east_cross(xg,cfg.DOMAIN_EAST)
            raw_reference=closest_distance_metrics(
                resample_arclength(xa,cfg.RESAMPLE_SPACING_KM),
                resample_arclength(xg,cfg.RESAMPLE_SPACING_KM))['D_mean_km']
            xa,move_a=light_spatial_smooth(xa,cfg.SPATIAL_SMOOTH_SIGMA_KM,cfg.RESAMPLE_SPACING_KM,cfg.MAX_SMOOTH_DISPLACEMENT_KM)
            xg,move_g=light_spatial_smooth(xg,cfg.SPATIAL_SMOOTH_SIGMA_KM,cfg.RESAMPLE_SPACING_KM,cfg.MAX_SMOOTH_DISPLACEMENT_KM)
            d=closest_distance_metrics(xa,xg)
            daily.append({'date':dayid,'year':day.year,'status':'ok',**d,
                          'adt_smooth_max_move_km':move_a,'glorys_smooth_max_move_km':move_g,
                          'D_raw_common_mean_km':raw_reference,
                          'smoothing_delta_D_km':d['D_mean_km']-raw_reference})
            for sec in SECTION_ORDER:
                ha=section_crossings(xa,sec);hg=section_crossings(xg,sec)
                uniq=(len(ha)==1 and len(hg)==1)
                section.append({'date':dayid,'year':day.year,'section':sec,
                    'adt_crossings':len(ha),'glorys_crossings':len(hg),
                    'ADT_radial_km':ha[0] if uniq else np.nan,
                    'GLORYS_radial_km':hg[0] if uniq else np.nan,
                    'offset_signed_km':(hg[0]-ha[0]) if uniq else np.nan,
                    'D_section_abs_km':abs(hg[0]-ha[0]) if uniq else np.nan,
                    'status':'unique' if uniq else 'ambiguous_or_absent'})
        except Exception as ex:
            daily.append({'date':dayid,'year':day.year,'status':'error','failure_reason':str(ex)})
            for sec in SECTION_ORDER:
                section.append({'date':dayid,'year':day.year,'section':sec,'status':'path_error',
                                'D_section_abs_km':np.nan,'offset_signed_km':np.nan})
        if idx%300==0:
            print(f'[SMOOTHED] {idx}/{len(dates)} dates; newly computed {len(daily)}')
            pd.DataFrame(old_d+daily).to_csv(oldfile,index=False)
            pd.DataFrame(old_s+section).to_csv(oldsec,index=False)
            cache_sig.write_text(chash,encoding='utf-8')
    dd=pd.DataFrame(old_d+daily).drop_duplicates('date',keep='last').sort_values('date')
    ss=pd.DataFrame(old_s+section).drop_duplicates(['date','section'],keep='last').sort_values(['date','section'])
    dd.to_csv(oldfile,index=False);ss.to_csv(oldsec,index=False)
    cache_sig.write_text(chash,encoding='utf-8')
    ok=dd[dd.status=='ok']
    print(f'[SMOOTHED] success: {len(ok)} / {len(dd)}')
    if len(ok)==0:raise RuntimeError('All axis NPZ comparisons failed. See first error: '+str(dd.failure_reason.dropna().head().tolist()))
    if len(ok)!=len(dd) and cfg.STRICT_FILES:
        raise RuntimeError(f'{len(dd)-len(ok)} paired dates failed; inspect {oldfile} before considering plots publishable')
    return ok.copy(),ss.copy(),'smoothed NPZ; common 144.3E'


def measure_archived(cfg,root,out):
    z72=root/cfg.R72_RESULT_NAME;z70=root/cfg.R70_RESULT_NAME
    for z in (z72,z70):
        if not z.is_file():raise FileNotFoundError('Missing archive: '+str(z))
    a=safe_zip_csv(z72,'tables/02_完整共域逐日误差.csv')
    a=a[a.status=='ok'].copy()
    daily=a[['date','year','common_mae_km','common_rmse_km','common_p90_km']].rename(columns={
         'common_mae_km':'D_mean_km','common_rmse_km':'D_RMSE_km','common_p90_km':'D_P90_km'})
    section=safe_zip_csv(z70,'daily/六断面逐日ADT与GLORYS成对误差.csv')
    section=section.rename(columns={'adt_crossings':'adt_crossings','glorys_crossings':'glorys_crossings'})
    section['offset_signed_km']=section['diff_km']
    section['D_section_abs_km']=np.where((section.adt_crossings==1)&(section.glorys_crossings==1),
                                        np.abs(section['diff_km']),np.nan)
    section['status']=np.where(section['D_section_abs_km'].notna(),'unique','ambiguous_or_absent')
    if cfg.MAX_DAYS:
        allow=set(daily.date.iloc[:cfg.MAX_DAYS]);daily=daily[daily.date.isin(allow)];section=section[section.date.isin(allow)]
    # Do not mutate the original archived source; always create separate reports.
    daily.to_csv(out/'tables'/'01_archived_original_daily_path_distance.csv',index=False)
    section.to_csv(out/'tables'/'02_archived_original_daily_sections.csv',index=False)
    return daily,section,'original R7.2/R7.0 archived values, NO SPATIAL SMOOTHING'


def choose_font(lang):
    if lang=='en':return None
    for fname in ['/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
                  '/usr/share/fonts/truetype/wqy/wqy-microhei.ttc',
                  '/usr/share/fonts/truetype/arphic/uming.ttc']:
        if Path(fname).is_file():
            font_manager.fontManager.addfont(fname)
            return font_manager.FontProperties(fname=fname)
    for name in ['Microsoft YaHei','Noto Sans CJK SC','Noto Sans CJK JP','SimHei','WenQuanYi Micro Hei']:
        if name in {f.name for f in font_manager.fontManager.ttflist}:
            return font_manager.FontProperties(family=name)
    warnings.warn('No CJK font found. English fallback is used. Install fonts-noto-cjk for Chinese text.')
    return None


def weighted_annual(daily, cfg):
    x=daily.copy()
    x.date=pd.to_datetime(x.date)
    x=x.dropna(subset=['D_mean_km'])
    groups=[]
    for yr,g in x.groupby(x.date.dt.year):
        r=g.D_mean_km.to_numpy(float)
        full=366 if pd.Timestamp(year=yr,month=12,day=31).is_leap_year else 365
        groups.append({'year':int(yr),'n_valid':len(r),'n_calendar':full,
            'coverage':len(r)/full,'D_mean_km':float(np.mean(r)),
            'D_median_km':float(np.median(r)),'P25':float(np.quantile(r,.25)),
            'P75':float(np.quantile(r,.75)),'P95':float(np.quantile(r,.95)),
            'sufficient':len(r)>=cfg.MIN_ANNUAL_DAYS})
    return pd.DataFrame(groups)


def monthly_path(daily,cfg):
    x=daily.copy()
    x.date=pd.to_datetime(x.date)
    x=x.set_index('date').sort_index()
    m=x.D_mean_km.resample('MS').agg(['count','mean','median'])
    m[['mean','median']]=m[['mean','median']].where(m['count']>=cfg.MIN_MONTH_DAYS)
    return m


def prepare_figure_tables(daily,sec,cfg,out):
    daily['date']=pd.to_datetime(daily['date'])
    sec['date']=pd.to_datetime(sec['date'])
    daily=daily.dropna(subset=['D_mean_km']).sort_values('date').copy()
    sec=sec[sec['date'].isin(set(daily.date))].copy()
    annual=weighted_annual(daily,cfg)
    monthly=monthly_path(daily,cfg)
    annual.to_csv(out/'tables'/'03_yearly_Dt_summary_and_coverage.csv',index=False)
    monthly.to_csv(out/'tables'/'04_monthly_Dt_summary.csv')
    n=sec.groupby('section',observed=True).agg(
        total_days=('date','nunique'),valid_unique=('D_section_abs_km','count'),
        abs_mean_km=('D_section_abs_km','mean'),
        abs_median_km=('D_section_abs_km','median'),
        abs_P95_km=('D_section_abs_km',lambda x:float(x.quantile(.95))))
    n['coverage']=n.valid_unique/n.total_days
    n.to_csv(out/'tables'/'05_six_section_coverage_and_ambiguity.csv')
    return daily,sec,annual,monthly,n


def plot_all(daily,sec,annual,monthly,coverage,origin,cfg,out):
    zh=(cfg.LANG=='zh');font=choose_font(cfg.LANG)
    if zh and font is not None:
        plt.rcParams['font.family']='sans-serif'
        plt.rcParams['font.sans-serif']=[font.get_name(), 'DejaVu Sans']
    plt.rcParams.update({'font.size':13,'axes.titlesize':15,'axes.labelsize':13,
                         'xtick.labelsize':12,'ytick.labelsize':12,'figure.facecolor':'white',
                         'savefig.facecolor':'white','axes.unicode_minus':False,'pdf.fonttype':42})
    color0=plt.rcParams['axes.prop_cycle'].by_key()['color'][0]
    color1=plt.rcParams['axes.prop_cycle'].by_key()['color'][1]
    color2=plt.rcParams['axes.prop_cycle'].by_key()['color'][2]
    fig,axs=plt.subplots(3,1,figsize=(14.6,15.4),gridspec_kw={'height_ratios':[1,1.06,1.2]})
    fig.subplots_adjust(left=.105,right=.973,top=.963,bottom=.061,hspace=.43)

    # a: full sample KDE + histogram, calibrated median/P95
    ax=axs[0]
    v=daily.D_mean_km.to_numpy(float)
    med=float(np.median(v));p95=float(np.quantile(v,.95))
    xmax=float(np.quantile(v,.995));xmax=max(xmax,p95*1.12,med*1.6)
    nbins=min(55,max(25,int(np.sqrt(len(v))/2)))
    bins=np.linspace(0,xmax,nbins)
    binwidth=float(bins[1]-bins[0])
    ax.hist(v,bins=bins,weights=np.full(len(v),1.0/(len(v)*binwidth)),
            density=False,alpha=.19,color=color0,edgecolor='none')
    if len(v)>3 and np.std(v)>1e-8:
        xs=np.linspace(0,xmax,450);density=gaussian_kde(v)(xs)
        ax.plot(xs,density,lw=2.8,color=color0)
        ax.fill_between(xs,density,alpha=.1,color=color0)
    ax.axvline(med,ls='--',lw=2.1,color=color1)
    ax.axvline(p95,ls='--',lw=2.1,color=color2)
    ax.text(med,ax.get_ylim()[1]*.90, f'Median = {med:.1f} km',rotation=90,va='top',ha='right',fontsize=12,color=color1)
    ax.text(p95,ax.get_ylim()[1]*.90, f'P95 = {p95:.1f} km',rotation=90,va='top',ha='right',fontsize=12,color=color2)
    ax.set(xlim=(0,xmax),xlabel=('逐日全路径双向最近平均距离 D(t)（km）' if zh else 'Daily bidirectional nearest-path mean distance D(t) (km)'),
           ylabel=('概率密度' if zh else 'Probability density'))
    ax.set_title(('(a) 逐日全路径距离的统计分布' if zh else '(a) Daily full-path distance distribution'),loc='left',fontweight='bold',pad=10)
    ax.grid(axis='y',alpha=.2,linestyle='--')

    # b: six section median/box with explicit n and NO multiple intersections
    ax=axs[1]
    samples=[sec.loc[sec.section==s,'D_section_abs_km'].dropna().to_numpy(float) for s in SECTION_ORDER]
    boxes=[]
    for x in samples:
        if len(x):
            boxes.append(dict(med=float(np.median(x)),q1=float(np.quantile(x,.25)),q3=float(np.quantile(x,.75)),
                              whislo=float(np.quantile(x,.05)),whishi=float(np.quantile(x,.95)),fliers=[]))
        else:
            boxes.append(dict(med=np.nan,q1=np.nan,q3=np.nan,whislo=np.nan,whishi=np.nan,fliers=[]))
    ax.bxp(boxes,positions=np.arange(1,7),widths=.52,showfliers=False,patch_artist=True,
           boxprops=dict(facecolor=color0,alpha=.17,edgecolor=color0,lw=1.7),
           medianprops=dict(color=color0,lw=2.5),
           whiskerprops=dict(color=color0,lw=1.4),capprops=dict(color=color0,lw=1.4))
    ax.set_xticks(range(1,7),labels=SECTION_ORDER)
    ax.set_ylabel('六断面交点径向距离差（km）' if zh else 'Section intersection offset (km)')
    ax.set_title('(b) 六断面逐日交点距离差分布' if zh else '(b) Daily section intersection offsets',loc='left',fontweight='bold',pad=10)
    ax.grid(axis='y',alpha=.20,linestyle='--')
    top=ax.get_ylim()[1];bot=ax.get_ylim()[0];height=top-bot
    for i,x in enumerate(samples,1):
        ax.text(i,-.105,f'n={len(x):,}',transform=ax.get_xaxis_transform(),ha='center',va='top',fontsize=10.5,alpha=.85,clip_on=False)

    # c: daily raw faint, 90-d smoothed display; yearly mean and 25-75% band
    ax=axs[2]
    g=daily.set_index('date').D_mean_km.sort_index()
    calendar=g.resample('D').mean()
    rolling=calendar.rolling(cfg.DISPLAY_ROLL_DAYS,min_periods=cfg.DISPLAY_ROLL_MIN_DAYS).mean()
    ax.plot(calendar.index,calendar.values,color=color0,lw=.45,alpha=.11,label='Daily raw' if not zh else '逐日原始误差')
    ax.plot(rolling.index,rolling.values,color=color0,lw=1.7,alpha=.86,label='90-day rolling mean' if not zh else '90天滑动平均')
    sufficient=annual[annual.sufficient]
    years=pd.to_datetime(sufficient.year.astype(str)+'-07-01')
    ax.plot(years,sufficient.D_mean_km.values,color=color1,marker='o',ms=5,lw=2.15,
            label='Annual mean (adequate coverage)' if not zh else '年度均值（有效覆盖）')
    ax.fill_between(years,sufficient.P25.values,sufficient.P75.values,color=color1,alpha=.12,
                    label='Annual P25–P75' if not zh else '年度P25–P75')
    few=annual[~annual.sufficient]
    if len(few):
        dat=pd.to_datetime(few.year.astype(str)+'-07-01')
        ax.scatter(dat,few.D_mean_km,marker='o',s=72,facecolors='none',edgecolors=color2,lw=2,
                   label='Low coverage (not full-year)' if not zh else '覆盖不足（非全年代表）',zorder=5)
    # Display up to approximately the daily P99, retain all raw observations in CSV and statistics.
    # Otherwise a few identification outliers (hundreds of km) flatten 31 years of ordinary evolution.
    ymax=max(float(np.quantile(calendar.dropna(),.99))*1.12,
             float(rolling.dropna().quantile(.99))*1.18,
             float(annual['P75'].quantile(.98))*1.18)
    extreme=calendar[calendar>ymax]
    if len(extreme):
        ax.scatter(extreme.index,np.full(len(extreme),.982*ymax),marker='|',
                   color=color2,s=24,alpha=.38,zorder=4)
    ax.set_ylim(0,ymax)
    ax.text(.993,.955,(f'超显示范围 {len(extreme)} 天（数值均保留）' if zh else
                         f'{len(extreme)} days above display range; all retained'),
            transform=ax.transAxes,ha='right',va='top',fontsize=10.5)
    ax.set(xlabel=('年份' if zh else 'Year'),ylabel=('D(t)，双向平均距离（km）' if zh else 'D(t), bidirectional mean distance (km)'))
    ax.set_title('(c) D(t)的多年时间演变' if zh else '(c) Long-term evolution of D(t)',loc='left',fontweight='bold',pad=10)
    ax.legend(loc='upper left',ncol=2,fontsize=10.4,frameon=False)
    ax.grid(alpha=.2,linestyle='--')
    if font is not None:
        for a in axs:
            for text in [a.title,a.xaxis.label,a.yaxis.label]:text.set_fontproperties(font)
        for legend in [ax.get_legend()]:
            if legend:
                for t in legend.get_texts():t.set_fontproperties(font)
    # Inset source transparency and methodology
    smoothing_label=('轻度空间平滑，σ≈10 km' if cfg.MODE=='smooth_npz' else '原始归档数据：未做空间平滑') if zh else (
                   'light spatial smoothing, sigma=10 km' if cfg.MODE=='smooth_npz' else 'archived data: NO spatial smoothing')
    foot=(f'ADT / GLORYS 50 m | 共域 144.3°E | {smoothing_label} | 2023 年覆盖不足 | (a)仅显示至P99.5' if zh else
          f'ADT / GLORYS 50 m | common 144.3E | {smoothing_label} | incomplete 2023 | (a) to P99.5')
    fig.text(.50,.018,foot,ha='center',fontsize=11,fontproperties=font)
    base='Figure_R79_ADT_GLORYS_ThreePanels_'+('ZH' if zh else 'EN')
    fig.savefig(out/'figures'/(base+'.png'),dpi=cfg.DPI)
    fig.savefig(out/'figures'/(base+'.pdf'))
    plt.close(fig)

    # Supplementary 6-section monthly trajectories
    fig,axes=plt.subplots(3,2,figsize=(15.5,12.6))
    fig.subplots_adjust(left=.09,right=.97,top=.935,bottom=.105,wspace=.23,hspace=.35)
    for ax,s in zip(axes.ravel(),SECTION_ORDER):
        x=sec[sec.section==s].set_index('date').D_section_abs_km.sort_index()
        x=x.groupby(level=0).mean().resample('D').mean()
        sm=x.rolling('90D',min_periods=45).mean()
        ax.plot(x.index,x.values,alpha=.13,lw=.5,color=color0)
        ax.plot(sm.index,sm.values,alpha=.92,lw=1.8,color=color0)
        ax.set_title(('断面 ' if zh else 'Section ')+s,loc='left',fontweight='bold')
        ax.set_ylabel('交点距离差（km）' if zh else 'Offset (km)')
        ax.set_xlabel('年份' if zh else 'Year')
        ax.grid(alpha=.17,ls='--')
        if font is not None:
            for txt in (ax.title,ax.xaxis.label,ax.yaxis.label):txt.set_fontproperties(font)
    fig.suptitle('六断面交点距离差逐日变化（90天平滑仅用于显示）' if zh else
                 'Daily section offset evolution (90-day average for display only)',
                 fontsize=17,fontweight='bold',fontproperties=font)
    fig.savefig(out/'figures'/('Figure_R79_SixSections_TimeSeries_'+('ZH' if zh else 'EN')+'.png'),dpi=cfg.DPI)
    fig.savefig(out/'figures'/('Figure_R79_SixSections_TimeSeries_'+('ZH' if zh else 'EN')+'.pdf'))
    plt.close(fig)
    return base


def save_report(cfg, root,out,daily,section,annual,coverage,origin):
    report={'dataset':'Satellite ADT (surface gradient-based path) vs GLORYS12V1 50 m velocity-core axis',
            'sources':origin,'config':asdict(cfg),'paired_path_days':int(len(daily)),
            'smoothing_delta_mean_km':float(daily['smoothing_delta_D_km'].mean()) if 'smoothing_delta_D_km' in daily else None,
            'smoothing_delta_abs_P95_km':float(daily['smoothing_delta_D_km'].abs().quantile(.95)) if 'smoothing_delta_D_km' in daily else None,
            'D_mean_km':float(daily.D_mean_km.mean()),
            'daily_D_median_km':float(daily.D_mean_km.median()),
            'daily_D_P95_km':float(daily.D_mean_km.quantile(.95)),
            'section_uniqueness_coverage':{s:{'valid_unique':int(coverage.loc[s,'valid_unique']),
                              'total_days':int(coverage.loc[s,'total_days'])} for s in SECTION_ORDER},
            'low_annual_coverage':annual.loc[~annual.sufficient,['year','n_valid','n_calendar']].to_dict('records'),
            'interpretation_limits':[ 'D(t) is symmetric mean nearest-polyline distance, not model forecast RMSE.',
                'Section offset uses ABSOLUTE difference of two unique radial intersection distances.',
                'Products represent different depths/physical features; GLORYS may assimilate altimetry.',
                'Missing dates are never interpolated for statistics.',
                'Smoothed paths should be described as sensitivity estimates, not replaced raw benchmarks.']}
    (out/'reports'/'R79_statistics_and_method_contract.json').write_text(json.dumps(report,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
    if cfg.ZIP_RESULTS:
        target=root/'R79_上传给ChatGPT_ADT与GLORYS三联图及距离统计.zip'
        with zipfile.ZipFile(target,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
            for f in sorted(out.rglob('*')):
                if f.is_file():z.write(f,f.relative_to(out))
        h=hashlib.sha256(target.read_bytes()).hexdigest()
        (Path(str(target)+'.sha256')).write_text(h+'  '+target.name+'\n',encoding='utf-8')
        print('[ZIP]',target,'SHA256',h)
    return report


def run(cfg=None):
    cfg=cfg or Config()
    if cfg.MODE not in ('smooth_npz','archive_fast'):raise ValueError('MODE must be smooth_npz or archive_fast')
    root,out=prepare_sources(cfg)
    if cfg.MODE=='smooth_npz':daily,section,origin=measure_smoothed(cfg,root,out)
    else:daily,section,origin=measure_archived(cfg,root,out)
    daily,section,annual,monthly,coverage=prepare_figure_tables(daily,section,cfg,out)
    _=plot_all(daily,section,annual,monthly,coverage,origin,cfg,out)
    report=save_report(cfg,root,out,daily,section,annual,coverage,origin)
    print('[COMPLETE] N=',report['paired_path_days'],'mean D(km)=',round(report['D_mean_km'],3))
    print('[FIGURES]',out/'figures')
    return report


## 第 3 步：正式运行。运行前确认项目 D 盘在 WSL 下挂载至 `/mnt/d/`。

首次检查可将 `MAX_DAYS=24`，但正式出图应恢复 `None`；默认不降级成“仅从旧结果画图”。如只有旧 ZIP 可切换 `MODE='archive_fast'`，但那是**未平滑旧统计图**。

In [ ]:
cfg = Config(
    ROOT='/mnt/d/KuroshioDaily1993_2025',
    MODE='smooth_npz',                 # 推荐：从冻结轴线NPZ实际轻度平滑后重新算D(t)
    DOMAIN_EAST=144.3,                # R7.2共域截断
    SPATIAL_SMOOTH_SIGMA_KM=10.0,    # 沿流轴弧长高斯核宽度（km）
    MAX_SMOOTH_DISPLACEMENT_KM=12.0, # 单点最大允许移动距离（km）
    RESAMPLE_SPACING_KM=5.0,        # 重采样间隔（km）
    MIN_ANNUAL_DAYS=180,            # 少于该值的年度仅显示空心标记
    MAX_DAYS=None,                  # 首次自检可设24；正式全量必须None
    LANG='zh',                      # 中文图；英文设'en'
    DPI=240,
    STRICT_FILES=True,
)
report = run(cfg)
print('每日有效配对数量:',report['paired_path_days'])
print('空间平滑引起的D均值变化(km):',report.get('smoothing_delta_mean_km'))
